In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer
from scipy.stats import skew
import numpy as np
import re
import os
from datetime import timedelta

In [2]:
datadir = '/scratch/hydro4/users/kv25483/MetricEvaluation/Data/'

### Create one dataframe containing all events for each resolution

In [8]:
def safe_parse_datetimes(df, file, resolution):
    """Parse start/end times, returning None if parsing fails."""
    try:
        df['start_time'] = pd.to_datetime(df['start_time'], format='mixed')
        df['end_time']   = pd.to_datetime(df['end_time'],   format='mixed')
        return df
    except Exception as e:
        print(f"  ⚠️  Datetime parse failed for {resolution} in {file}: {e}")
        return None

# Track files that had to be skipped
skipped_files = []

files = np.sort(os.listdir(datadir + "DanishRainData_Outputs/5mins_new/"))[1:]
skip  = {"All_events_571820_precip_minute.csv", 'All_events_5781_svk_precip_minute.csv'}
print(f"Total files to process: {len(files)}")
print(f"Skipping: {skip}\n")

all_events_5mins  = []
all_events_10mins = []
all_events_30mins = []
all_events_60mins = []
total_events = 0


for num, file in enumerate(files):
    if file in skip:
        print(f"[{num}] Skipping {file}")
        continue

    df_5m  = pd.read_csv(datadir + f"DanishRainData_Outputs/5mins_new/{file}")
    df_10m = pd.read_csv(datadir + f"DanishRainData_Outputs/{10}mins_new/{file}")
    df_30m = pd.read_csv(datadir + f"DanishRainData_Outputs/{30}mins_new/{file}")
    df_60m = pd.read_csv(datadir + f"DanishRainData_Outputs/{60}mins_new/{file}")

    print(f"[{num}] {file}")
    print(f"  Loaded:  5m={len(df_5m)}  10m={len(df_10m)}  "
          f"30m={len(df_30m)}  60m={len(df_60m)}")

    # Clean gauge_num
    df_5m['gauge_num'] = df_5m['gauge_num'].apply(
        lambda x: int(x.split('_')[0]) if isinstance(x, str)
        and x.split('_')[0].isdigit() else x)

    # Parse datetimes safely — skip file if any resolution fails
    parse_failed = False
    for df, res in [(df_5m, '5m'), (df_10m, '10m'), (df_30m, '30m'), (df_60m, '60m')]:
        result = safe_parse_datetimes(df, file, res)
        if result is None:
            parse_failed = True
            break

    if parse_failed:
        print(f"  ⚠️  Skipping {file} due to datetime parse error")
        skipped_files.append(file)
        continue

    # Floor start times to hour for matching
    for df in [df_5m, df_10m, df_30m, df_60m]:
        df['start_hour'] = df['start_time'].dt.floor('h')

    common_starts = (
        set(df_5m['start_hour']) &
        set(df_10m['start_hour']) &
        set(df_30m['start_hour']) &
        set(df_60m['start_hour'])
    )

    print(f"  Before filter: 5m={len(df_5m)}  10m={len(df_10m)}  "
          f"30m={len(df_30m)}  60m={len(df_60m)}")
    print(f"  Common start times found: {len(common_starts)}")
    print(f"  Dropped:  5m={len(df_5m)-len(common_starts)}  "
          f"10m={len(df_10m)-len(common_starts)}  "
          f"30m={len(df_30m)-len(common_starts)}  "
          f"60m={len(df_60m)-len(common_starts)}")

    df_5m  = df_5m[df_5m['start_hour'].isin(common_starts)].drop(
        columns='start_hour').reset_index(drop=True)
    df_10m = df_10m[df_10m['start_hour'].isin(common_starts)].drop(
        columns='start_hour').reset_index(drop=True)
    df_30m = df_30m[df_30m['start_hour'].isin(common_starts)].drop(
        columns='start_hour').reset_index(drop=True)
    df_60m = df_60m[df_60m['start_hour'].isin(common_starts)].drop(
        columns='start_hour').reset_index(drop=True)

    lengths = [len(df_5m), len(df_10m), len(df_30m), len(df_60m)]
    if len(set(lengths)) != 1:
        print(f"  ⚠️  LENGTH MISMATCH after filter: {lengths}")
    else:
        print(f"  ✓  All resolutions agree: {lengths[0]} events retained")

    total_events += len(common_starts)
    print(f"  Running total: {total_events}")

    # Time alignment check
    dfs = {'5m': df_5m, '10m': df_10m, '30m': df_30m, '60m': df_60m}
    mismatches = 0
    for i in range(len(df_5m)):
        ref_start = df_5m.iloc[i]['start_time']
        ref_end   = df_5m.iloc[i]['end_time']
        for res, df in dfs.items():
            if abs(df.iloc[i]['start_time'] - ref_start) > timedelta(minutes=60):
                print(f"  ⚠️  START mismatch [{res}] index={i}  "
                      f"{res}:{df.iloc[i]['start_time']}  5m:{ref_start}")
                mismatches += 1
            if abs(df.iloc[i]['end_time'] - ref_end) > timedelta(minutes=60):
                print(f"  ⚠️  END mismatch [{res}] index={i}  "
                      f"{res}:{df.iloc[i]['end_time']}  5m:{ref_end}")
                mismatches += 1
    if mismatches == 0:
        print(f"  ✓  No time mismatches detected")
    else:
        print(f"  ⚠️  {mismatches} time mismatches detected")

    print(f"  First event: 5m={df_5m.iloc[0]['start_time']}  "
          f"10m={df_10m.iloc[0]['start_time']}  "
          f"30m={df_30m.iloc[0]['start_time']}  "
          f"60m={df_60m.iloc[0]['start_time']}")
    print(f"  Last event:  5m={df_5m.iloc[-1]['start_time']}  "
          f"10m={df_10m.iloc[-1]['start_time']}  "
          f"30m={df_30m.iloc[-1]['start_time']}  "
          f"60m={df_60m.iloc[-1]['start_time']}")

    all_events_5mins.append(df_5m)
    all_events_10mins.append(df_10m)
    all_events_30mins.append(df_30m)
    all_events_60mins.append(df_60m)

# Concatenate
all_events_df_5mins  = pd.concat(all_events_5mins).reset_index(drop=True)
all_events_df_10mins = pd.concat(all_events_10mins).reset_index(drop=True)
all_events_df_30mins = pd.concat(all_events_30mins).reset_index(drop=True)
all_events_df_60mins = pd.concat(all_events_60mins).reset_index(drop=True)

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Total events retained: {total_events}")
print(f"5m:  {len(all_events_df_5mins)}")
print(f"10m: {len(all_events_df_10mins)}")
print(f"30m: {len(all_events_df_30mins)}")
print(f"60m: {len(all_events_df_60mins)}")
all_equal = len(set([len(all_events_df_5mins), len(all_events_df_10mins),
                     len(all_events_df_30mins), len(all_events_df_60mins)])) == 1
print(f"All lengths equal: {'✓' if all_equal else '⚠️  MISMATCH'}")

if skipped_files:
    print(f"\nSkipped files ({len(skipped_files)}):")
    for f in skipped_files:
        print(f"  {f}")

Total files to process: 345
Skipping: {'All_events_5781_svk_precip_minute.csv', 'All_events_571820_precip_minute.csv'}

[0] All_events_500920_precip_minute.csv
  Loaded:  5m=944  10m=937  30m=921  60m=872
  Before filter: 5m=944  10m=937  30m=921  60m=872
  Common start times found: 869
  Dropped:  5m=75  10m=68  30m=52  60m=3
  ✓  All resolutions agree: 869 events retained
  Running total: 869
  ✓  No time mismatches detected
  First event: 5m=2012-04-06 00:15:00  10m=2012-04-06 00:10:00  30m=2012-04-06 00:00:00  60m=2012-04-06 00:00:00
  Last event:  5m=2024-12-30 02:05:00  10m=2024-12-30 02:00:00  30m=2024-12-30 02:00:00  60m=2024-12-30 02:00:00
[1] All_events_5012_svk_precip_minute.csv
  Loaded:  5m=293  10m=291  30m=271  60m=241
  Before filter: 5m=293  10m=291  30m=271  60m=241
  Common start times found: 241
  Dropped:  5m=52  10m=50  30m=30  60m=0
  ✓  All resolutions agree: 241 events retained
  Running total: 1110
  ✓  No time mismatches detected
  First event: 5m=1979-01-11 

[15] All_events_5057_svk_precip_minute.csv
  Loaded:  5m=2435  10m=2405  30m=2258  60m=1992
  Before filter: 5m=2435  10m=2405  30m=2258  60m=1992
  Common start times found: 1992
  Dropped:  5m=443  10m=413  30m=266  60m=0
  ✓  All resolutions agree: 1992 events retained
  Running total: 23663
  ✓  No time mismatches detected
  First event: 5m=1997-06-21 02:55:00  10m=1997-06-21 02:50:00  30m=1997-06-21 02:30:00  60m=1997-06-21 02:00:00
  Last event:  5m=2025-03-25 06:50:00  10m=2025-03-25 06:50:00  30m=2025-03-25 06:30:00  60m=2025-03-25 06:00:00
[16] All_events_5058_svk_precip_minute.csv
  Loaded:  5m=2391  10m=2352  30m=2198  60m=1965
  Before filter: 5m=2391  10m=2352  30m=2198  60m=1965
  Common start times found: 1965
  Dropped:  5m=426  10m=387  30m=233  60m=0
  ✓  All resolutions agree: 1965 events retained
  Running total: 25628
  ✓  No time mismatches detected
  First event: 5m=1997-09-06 12:55:00  10m=1997-09-06 12:50:00  30m=1997-09-06 12:30:00  60m=1997-09-06 12:00:00
  L

[30] All_events_5121_svk_precip_minute.csv
  Loaded:  5m=1780  10m=1780  30m=1671  60m=1490
  Before filter: 5m=1780  10m=1780  30m=1671  60m=1490
  Common start times found: 1490
  Dropped:  5m=290  10m=290  30m=181  60m=0
  ✓  All resolutions agree: 1490 events retained
  Running total: 42290
  ✓  No time mismatches detected
  First event: 5m=2005-09-27 13:05:00  10m=2005-09-27 13:00:00  30m=2005-09-27 13:00:00  60m=2005-09-27 13:00:00
  Last event:  5m=2025-03-12 16:20:00  10m=2025-03-12 16:20:00  30m=2025-03-12 16:00:00  60m=2025-03-12 16:00:00
[31] All_events_5122_svk_precip_minute.csv
  Loaded:  5m=1792  10m=1791  30m=1693  60m=1521
  Before filter: 5m=1792  10m=1791  30m=1693  60m=1521
  Common start times found: 1520
  Dropped:  5m=272  10m=271  30m=173  60m=1
  ✓  All resolutions agree: 1520 events retained
  Running total: 43810
  ✓  No time mismatches detected
  First event: 5m=2005-08-27 11:05:00  10m=2005-08-27 11:00:00  30m=2005-08-27 11:00:00  60m=2005-08-27 11:00:00
  L

[45] All_events_5163_svk_precip_minute.csv
  Loaded:  5m=298  10m=297  30m=277  60m=244
  Before filter: 5m=298  10m=297  30m=277  60m=244
  Common start times found: 244
  Dropped:  5m=54  10m=53  30m=33  60m=0
  ✓  All resolutions agree: 244 events retained
  Running total: 53496
  ✓  No time mismatches detected
  First event: 5m=2021-11-26 21:00:00  10m=2021-11-26 21:00:00  30m=2021-11-26 21:00:00  60m=2021-11-26 21:00:00
  Last event:  5m=2025-03-11 12:35:00  10m=2025-03-11 12:30:00  30m=2025-03-11 12:30:00  60m=2025-03-11 12:00:00
[46] All_events_5164_svk_precip_minute.csv
  Loaded:  5m=315  10m=314  30m=296  60m=259
  Before filter: 5m=315  10m=314  30m=296  60m=259
  Common start times found: 259
  Dropped:  5m=56  10m=55  30m=37  60m=0
  ✓  All resolutions agree: 259 events retained
  Running total: 53755
  ✓  No time mismatches detected
  First event: 5m=2021-11-26 20:45:00  10m=2021-11-26 20:40:00  30m=2021-11-26 20:30:00  60m=2021-11-26 20:00:00
  Last event:  5m=2025-03-11 

[60] All_events_5192_svk_precip_minute.csv
  Loaded:  5m=3879  10m=3879  30m=3665  60m=3267
  Before filter: 5m=3879  10m=3879  30m=3665  60m=3267
  Common start times found: 3267
  Dropped:  5m=612  10m=612  30m=398  60m=0
  ✓  All resolutions agree: 3267 events retained
  Running total: 71568
  ✓  No time mismatches detected
  First event: 5m=1979-01-07 08:05:00  10m=1979-01-07 08:00:00  30m=1979-01-07 08:00:00  60m=1979-01-07 08:00:00
  Last event:  5m=2025-02-24 08:10:00  10m=2025-02-24 08:10:00  30m=2025-02-24 08:00:00  60m=2025-02-24 08:00:00
[61] All_events_5195_svk_precip_minute.csv
  Loaded:  5m=1398  10m=1397  30m=1317  60m=1185
  Before filter: 5m=1398  10m=1397  30m=1317  60m=1185
  Common start times found: 1185
  Dropped:  5m=213  10m=212  30m=132  60m=0
  ✓  All resolutions agree: 1185 events retained
  Running total: 72753
  ✓  No time mismatches detected
  First event: 5m=2009-09-28 04:15:00  10m=2009-09-28 04:10:00  30m=2009-09-28 04:00:00  60m=2009-09-28 04:00:00
  L

...

  ✓  No time mismatches detected
  First event: 5m=1979-01-24 15:00:00  10m=1979-01-24 15:00:00  30m=1979-01-24 15:00:00  60m=1979-01-24 15:00:00
  Last event:  5m=2025-03-13 12:40:00  10m=2025-03-13 12:40:00  30m=2025-03-13 12:30:00  60m=2025-03-13 12:00:00
[274] All_events_5815_svk_precip_minute.csv
  Loaded:  5m=2333  10m=2297  30m=2133  60m=1848
  Before filter: 5m=2333  10m=2297  30m=2133  60m=1848
  Common start times found: 1848
  Dropped:  5m=485  10m=449  30m=285  60m=0
  ✓  All resolutions agree: 1848 events retained
  Running total: 307064
  ✓  No time mismatches detected
  First event: 5m=1996-02-16 11:15:00  10m=1996-02-16 11:10:00  30m=1996-02-16 11:00:00  60m=1996-02-16 11:00:00
  Last event:  5m=2025-03-25 12:55:00  10m=2025-03-25 12:50:00  30m=2025-03-25 12:30:00  60m=2025-03-25 12:00:00
[275] All_events_5824_svk_precip_minute.csv
  Loaded:  5m=708  10m=694  30m=634  60m=543
  Before filter: 5m=708  10m=694  30m=634  60m=543
  Common start times found: 543
  Dropped:  

  ✓  No time mismatches detected
  First event: 5m=2012-11-03 20:50:00  10m=2012-11-03 20:50:00  30m=2012-11-03 20:30:00  60m=2012-11-03 20:00:00
  Last event:  5m=2025-03-13 10:00:00  10m=2025-03-13 10:00:00  30m=2025-03-13 10:00:00  60m=2025-03-13 10:00:00
[289] All_events_5879_svk_precip_minute.csv
  Loaded:  5m=1580  10m=1548  30m=1430  60m=1237
  Before filter: 5m=1580  10m=1548  30m=1430  60m=1237
  Common start times found: 1237
  Dropped:  5m=343  10m=311  30m=193  60m=0
  ✓  All resolutions agree: 1237 events retained
  Running total: 322539
  ✓  No time mismatches detected
  First event: 5m=1979-01-09 10:00:00  10m=1979-01-09 10:00:00  30m=1979-01-09 10:00:00  60m=1979-01-09 10:00:00
  Last event:  5m=2025-02-21 00:10:00  10m=2025-02-21 00:10:00  30m=2025-02-21 00:00:00  60m=2025-02-21 00:00:00
[290] All_events_588020_precip_minute.csv
  Loaded:  5m=988  10m=974  30m=938  60m=871
  Before filter: 5m=988  10m=974  30m=938  60m=871
  Common start times found: 869
  Dropped:  5m

[304] All_events_597020_precip_minute.csv
  Loaded:  5m=991  10m=970  30m=937  60m=831
  Before filter: 5m=991  10m=970  30m=937  60m=831
  Common start times found: 830
  Dropped:  5m=161  10m=140  30m=107  60m=1
  ✓  All resolutions agree: 830 events retained
  Running total: 340154
  ✓  No time mismatches detected
  First event: 5m=2012-03-30 20:05:00  10m=2012-03-30 20:00:00  30m=2012-03-30 20:00:00  60m=2012-03-30 20:00:00
  Last event:  5m=2024-12-22 19:55:00  10m=2024-12-22 19:50:00  30m=2024-12-22 19:30:00  60m=2024-12-22 19:00:00
[305] All_events_5975_svk_precip_minute.csv
  Loaded:  5m=322  10m=311  30m=283  60m=243
  Before filter: 5m=322  10m=311  30m=283  60m=243
  Common start times found: 243
  Dropped:  5m=79  10m=68  30m=40  60m=0
  ✓  All resolutions agree: 243 events retained
  Running total: 340397
  ✓  No time mismatches detected
  First event: 5m=1993-11-13 14:40:00  10m=1993-11-13 14:40:00  30m=1993-11-13 14:30:00  60m=1993-11-13 14:00:00
  Last event:  5m=1998-0

  ✓  No time mismatches detected
  First event: 5m=2015-03-20 08:10:00  10m=2015-03-20 08:10:00  30m=2015-03-20 08:00:00  60m=2015-03-20 08:00:00
  Last event:  5m=2024-12-30 04:45:00  10m=2024-12-30 04:40:00  30m=2024-12-30 04:30:00  60m=2024-12-30 04:00:00
[320] All_events_607000_precip_minute.csv
  Loaded:  5m=1143  10m=1125  30m=1087  60m=1021
  Before filter: 5m=1143  10m=1125  30m=1087  60m=1021
  Common start times found: 1018
  Dropped:  5m=125  10m=107  30m=69  60m=3
  ✓  All resolutions agree: 1018 events retained
  Running total: 353214
  ✓  No time mismatches detected
  First event: 5m=2012-04-09 00:05:00  10m=2012-04-09 00:00:00  30m=2012-04-09 00:00:00  60m=2012-04-09 00:00:00
  Last event:  5m=2024-12-24 16:40:00  10m=2024-12-24 16:40:00  30m=2024-12-24 16:30:00  60m=2024-12-24 16:00:00
[321] All_events_607200_precip_minute.csv
  Loaded:  5m=72  10m=69  30m=68  60m=63
  Before filter: 5m=72  10m=69  30m=68  60m=63
  Common start times found: 62
  Dropped:  5m=10  10m=7  

[335] All_events_615600_precip_minute.csv
  Loaded:  5m=73  10m=72  30m=68  60m=64
  Before filter: 5m=73  10m=72  30m=68  60m=64
  Common start times found: 64
  Dropped:  5m=9  10m=8  30m=4  60m=0
  ✓  All resolutions agree: 64 events retained
  Running total: 356851
  ✓  No time mismatches detected
  First event: 5m=2024-03-05 17:55:00  10m=2024-03-05 17:50:00  30m=2024-03-05 17:30:00  60m=2024-03-05 17:00:00
  Last event:  5m=2024-12-24 18:30:00  10m=2024-12-24 18:30:00  30m=2024-12-24 18:30:00  60m=2024-12-24 18:00:00
[336] All_events_615900_precip_minute.csv
  Loaded:  5m=703  10m=694  30m=671  60m=641
  Before filter: 5m=703  10m=694  30m=671  60m=641
  Common start times found: 641
  Dropped:  5m=62  10m=53  30m=30  60m=0
  ✓  All resolutions agree: 641 events retained
  Running total: 357492
  ✓  No time mismatches detected
  First event: 5m=2016-06-15 17:10:00  10m=2016-06-15 17:10:00  30m=2016-06-15 17:00:00  60m=2016-06-15 17:00:00
  Last event:  5m=2024-12-22 18:20:00  10m

In [30]:
all_events_df_5mins.to_csv(datadir + 'IntermediateFiles/all_events_df_5mins_new.csv', index=False)
all_events_df_10mins.to_csv(datadir +  'IntermediateFiles/all_events_df_10mins_new.csv', index=False)
all_events_df_30mins.to_csv(datadir + 'IntermediateFiles/all_events_df_30mins_new.csv', index=False)
all_events_df_60mins.to_csv(datadir + 'IntermediateFiles/all_events_df_60mins_new.csv', index=False)

### Scale and transform

In [38]:
all_events_df_5mins = pd.read_csv(datadir +  'IntermediateFiles/all_events_df_5mins_new.csv')
all_events_df_10mins = pd.read_csv(datadir +  'IntermediateFiles/all_events_df_10mins_new.csv')
all_events_df_30mins = pd.read_csv(datadir + 'IntermediateFiles/all_events_df_30mins_new.csv')
all_events_df_60mins = pd.read_csv(datadir + 'IntermediateFiles/all_events_df_60mins_new.csv')

In [ ]:
# all_events_df_5mins = all_events_df_5mins.rename(columns={'lorentz_asymetry': 'lorenz_asymmetry'})
# all_events_df_10mins = all_events_df_10mins.rename(columns={'lorentz_asymetry': 'lorenz_asymmetry'})
# all_events_df_30mins = all_events_df_30mins.rename(columns={'lorentz_asymetry': 'lorenz_asymmetry'})
# all_events_df_60mins = all_events_df_60mins.rename(columns={'lorentz_asymetry': 'lorenz_asymmetry'})

# all_events_df_5mins = all_events_df_5mins.rename(columns={'lorentz_asymetry_DMC_10': 'lorenz_asymmetry_DMC_10'})
# all_events_df_10mins = all_events_df_10mins.rename(columns={'lorentz_asymetry_DMC_10': 'lorenz_asymmetry_DMC_10'})
# all_events_df_30mins = all_events_df_30mins.rename(columns={'lorentz_asymetry_DMC_10': 'lorenz_asymmetry_DMC_10'})
# all_events_df_60mins = all_events_df_60mins.rename(columns={'lorentz_asymetry_DMC_10': 'lorenz_asymmetry_DMC_10'})

### Remove rows with unrealistically high precip

In [39]:
bad_indices = all_events_df_5mins.sort_values(by="total_precip", ascending=False)[:15].index

# Drop rows at these indices from all dataframes
all_events_df_5mins = all_events_df_5mins.drop(index=bad_indices)
all_events_df_10mins = all_events_df_10mins.drop(index=bad_indices)
all_events_df_30mins = all_events_df_30mins.drop(index=bad_indices)
all_events_df_60mins = all_events_df_60mins.drop(index=bad_indices)

for df in [all_events_df_5mins, all_events_df_10mins, all_events_df_30mins, all_events_df_60mins]:
    df.reset_index(drop=True, inplace=True)
    
print(len(all_events_df_5mins), len(all_events_df_10mins), len(all_events_df_30mins), len(all_events_df_60mins))

230607 230618 230638 230622


### Remove very small events

In [40]:
print(len(all_events_df_5mins))
all_events_df_5mins = all_events_df_5mins[all_events_df_5mins['total_precip']>4].copy()
print(len(all_events_df_5mins))

print(len(all_events_df_10mins))
all_events_df_10mins = all_events_df_10mins[all_events_df_10mins['total_precip']>4].copy()
print(len(all_events_df_10mins))

print(len(all_events_df_30mins))
all_events_df_30mins = all_events_df_30mins[all_events_df_30mins['total_precip']>4].copy()
print(len(all_events_df_30mins))

print(len(all_events_df_60mins))
all_events_df_60mins = all_events_df_60mins[all_events_df_60mins['total_precip']>4].copy()
print(len(all_events_df_60mins))

230607
230607
230618
230618
230638
230638
230622
230622


# Create version for all resolutions, just raw
### Specify just raw columns to keep

In [41]:
raw_cols = []
for col in all_events_df_5mins.columns:
    if not col.endswith('_dmc') and not col.endswith('dblnorm'):
        raw_cols.append(col)

### Also remove 'frac_q1', 'frac_q2', 'frac_q3', 'frac_q4'
Because these bring lots of nans (with current calculation method)  
ni is the same as peak mean ratio

In [58]:
cols_to_del = ['frac_q1', 'frac_q2', 'frac_q3', 'frac_q4', 'BSC', 'ni', 'm1', 'm2', 'm3', 'm4', 'm5']
more_cols_to_del = ['gauge_num',  'start_time', 'end_time', 'min_intensity_raw', 'heaviest_half_raw',
                    'T50_raw', 'BSC_Index_raw', 'peak_mean_ratio_scaled_raw',"relative_amp_scaled_raw"]
                   
raw_cols = [x for x in raw_cols if x not in cols_to_del]  
raw_cols = [x for x in raw_cols if x not in more_cols_to_del]   

In [43]:
all_events_df_5mins_raw = all_events_df_5mins[raw_cols].copy()
all_events_df_10mins_raw = all_events_df_10mins[raw_cols].copy()
all_events_df_30mins_raw = all_events_df_30mins[raw_cols].copy()
all_events_df_60mins_raw = all_events_df_60mins[raw_cols].copy()

In [44]:
all_events_df_60mins_raw["I30"] = 0

In [45]:
print(len(all_events_df_5mins_raw), len(all_events_df_10mins_raw), len(all_events_df_30mins_raw), len(all_events_df_60mins_raw))

bad_indices_all = []
for all_events_df in [all_events_df_5mins_raw, all_events_df_10mins_raw, all_events_df_30mins_raw, all_events_df_60mins_raw]:
    columns_with_nan = all_events_df.columns[all_events_df.isnull().any()].tolist()
    print("Columns with at least one NaN value:")
    print(columns_with_nan)
    # #Find indices of rows with any NaNs in the 60-minute data
    bad_indices = all_events_df[all_events_df.isnull().any(axis=1)].index
    bad_indices_lst = bad_indices.tolist()
    print(len(bad_indices_lst))
    if len(bad_indices) >0:
        bad_indices_all.extend(bad_indices_lst)
        bad_indices_all.extend(bad_indices_lst)
        bad_indices_all.extend(bad_indices_lst)  # keep this the same
    
# Deduplicate before dropping
bad_indices_all = list(set(bad_indices_all))

# Drop rows at these indices from all dataframes
all_events_df_5mins_raw  = all_events_df_5mins_raw.drop(index=bad_indices_all)
all_events_df_10mins_raw = all_events_df_10mins_raw.drop(index=bad_indices_all)
all_events_df_30mins_raw = all_events_df_30mins_raw.drop(index=bad_indices_all)
all_events_df_60mins_raw = all_events_df_60mins_raw.drop(index=bad_indices_all)

print(len(all_events_df_5mins_raw), len(all_events_df_10mins_raw), len(all_events_df_30mins_raw), len(all_events_df_60mins_raw))

230607 230618 230638 230622
Columns with at least one NaN value:
['event_num']
43452
Columns with at least one NaN value:
[]
0
Columns with at least one NaN value:
[]
0
Columns with at least one NaN value:
[]
0
187155 187166 187186 187170


### Join all into one dataframe

In [46]:
metric_columns =all_events_df_5mins_raw.columns

In [47]:
all_events_df_5mins_raw = all_events_df_5mins_raw.add_suffix('_5m')
all_events_df_10mins_raw = all_events_df_10mins_raw.add_suffix('_10m')
all_events_df_30mins_raw = all_events_df_30mins_raw.add_suffix('_30m')
all_events_df_60mins_raw = all_events_df_60mins_raw.add_suffix('_60m')

In [48]:
all_events_all_res = pd.concat([all_events_df_5mins_raw, all_events_df_10mins_raw,
                               all_events_df_30mins_raw, all_events_df_60mins_raw], axis=1)
# Step 2: Define the event number columns from each resolution
event_cols = ['event_num_5m', 'event_num_10m', 'event_num_30m', 'event_num_60m']

# Step 3: Drop rows with any missing event numbers
all_events_all_res = all_events_all_res.dropna(subset=event_cols)

# Step 4: Keep only rows where all event numbers are equal (same event matched across resolutions)
all_events_all_res = all_events_all_res[all_events_all_res[event_cols].nunique(axis=1) == 1]

In [49]:
all_events_all_res.to_csv(datadir + "IntermediateFiles/NotScaled_AllRes_new.csv",index=False)

# Create version for 5 minutes, DMC and raw
### Specify columns to keep

In [62]:
cols_to_del = ['frac_q1', 'frac_q2', 'frac_q3', 'frac_q4', 'BSC', 'ni', 'T50', 'm1', 'm2', 'm3', 'm4', 'm5', 'BSC_Index']
more_cols_to_del = ['gauge_num', 'start_time', 'end_time', 'event_num', 'heaviest_half_raw']

all_to_del = cols_to_del + more_cols_to_del
all_columns = df.columns.tolist()
cols_to_keep = [col for col in all_columns if col not in all_to_del]

# Remove any column that matches exactly or starts with one of the names (e.g., has suffixes like _DMC_10)
cols_to_keep = [col for col in cols_to_keep if col not in all_to_del]

# Then filter the DataFrame
all_events_df_5mins_filtered = all_events_df_5mins[cols_to_keep].copy()

In [63]:
all_events_df_5mins_filtered['mean_intensity_dmc'] = all_events_df_5mins_filtered['mean_intensity_dmc']/10

In [64]:
all_events_df_5mins_filtered.to_csv(datadir + "IntermediateFiles/NotScaled_RawVsDMC_new.csv",index=False)